# 构建图像生成应用（Azure OpenAI）

大型语言模型不仅仅是文本生成。你还可以根据文本描述生成图像。图像作为一种模态，在医疗技术、建筑、旅游、游戏开发、营销等领域都有用处。在本课中，我们使用微软Foundry上的当今<strong>GPT图像</strong>模型。

## 学习目标

通过本课，你将能：

- 解释什么是图像生成及其用途。
- 了解`gpt-image`模型家族及其与传统DALL·E模型的区别。
- 构建图像生成应用并编辑图像。

## 什么是图像生成？

图像生成模型根据文本提示创建图像。现代模型如`gpt-image`在训练期间学习文本与图像的关系，然后逐步将随机噪声转换成与你的提示相匹配的图像。

两个著名的图像模型家族是：

- **`gpt-image`（OpenAI）** — 本课使用的现今一代模型。它支持文本生成图像及图像编辑（带掩码的修补）。
- **Midjourney** — 一个流行的第三方模型，拥有自己的服务和基于Discord的工作流程。

> 旧版OpenAI图像模型—<strong>DALL·E 2</strong>和<strong>DALL·E 3</strong>—已成为传统版本。DALL·E 3不再用于新部署，且只有DALL·E 2支持`create_variation`等功能。新项目请使用`gpt-image`模型。

在微软Foundry上，**`gpt-image-2`**是最新且功能最强大的图像模型，是推荐的默认模型。`gpt-image-1.5`和`gpt-image-1-mini`也已通用。

> **重要提示：** `gpt-image`模型返回生成的图像为<strong>base64</strong>格式（`b64_json`），而非URL。代码需要解码base64字符串到字节并保存 — 不存在图像下载URL。


## 构建你的第一个图像生成应用程序

那么构建一个图像生成应用程序需要什么呢？你需要以下库：

- **python-dotenv**，强烈建议使用这个库将你的密钥保存在 *.env* 文件中，远离代码。
- **openai**，这个库是你与 OpenAI API 交互时使用的。
- **pillow**，用于在 Python 中处理图像。

如果还没有完成，按照[Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst)页面上的说明创建一个 Microsoft Foundry 资源和模型。选择 **gpt-image-2** 作为模型（最新的 Azure OpenAI 图像模型；DALL·E 是旧版本）。

1. 创建一个 *.env* 文件，内容如下：

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    在 Microsoft Foundry 门户中资源的“部署”部分找到这些信息。



1. 将上述库收集到一个名为 *requirements.txt* 的文件中，内容如下：

    ```text
    python-dotenv
    openai
    pillow
    ```


1. 接着，创建虚拟环境并安装这些库：


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> 对于 Windows，使用以下命令创建并激活虚拟环境：

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. 在名为 *app.py* 的文件中添加以下代码：

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # 导入 dotenv
    dotenv.load_dotenv()

    # 配置 Azure OpenAI（Microsoft Foundry）客户端。
    # 图像模型需要较新的 API 版本 — 请查看 Foundry 文档，了解您的模型所需的版本。
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # 使用图像生成 API 创建图像
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # 在这里输入您的提示文本
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # 例如 "gpt-image-2"
        )
        # 设置存储图像的目录
        image_dir = os.path.join(os.curdir, 'images')

        # 如果目录不存在，则创建它
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # 初始化图像路径（注意文件类型应为 png）
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image 模型以 base64（b64_json）格式返回图像，而不是 URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # 在默认图像查看器中显示图像
        image = Image.open(image_path)
        image.show()

    # 捕获异常
    except BadRequestError as err:
        print(err)

    ```

让我们解释这段代码：

- 首先，导入所需的库，包括 OpenAI 库、dotenv 库、base64 模块和 Pillow 库。

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- 接着，我们从 *.env* 文件加载环境变量。

    ```python
    # 导入dotenv
    dotenv.load_dotenv()
    ```

- 然后，我们配置 Azure OpenAI（Microsoft Foundry）客户端。

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- 接下来，我们生成图片：

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # 在此输入您的提示文本
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    `gpt-image` 模型返回的图像是以 **base64** 字符串形式存储在 `data[0].b64_json` 中。我们将其解码为字节并写入文件——没有下载用的 URL。

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- 最后，我们打开图片，使用标准图像查看器显示它：

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### 关于生成图像的更多细节

让我们看看 `images.generate` 的参数：

- **prompt**，用于生成图像的文本提示。这里是“Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils”。
- **size**，生成图像的大小（例如 `1024x1024`、`1536x1024`、`1024x1536` 或 `"auto"`）。
- **n**，生成图像的数量。这里生成一张。
- **model**，你的图像模型部署名称（例如 `gpt-image-2`）。

> 图像模型不接受 `temperature` 参数——那是文本生成的控制。要获得多样性，请再次调用 API；减少多样性，则让提示更具体。

## 图像生成的额外功能

你已经看到了如何用几行 Python 代码生成一张图像。`gpt-image` 模型还可以<strong>编辑</strong>现有图像。通过提供一张图片、一个可选的<strong>遮罩</strong>（标记要更改的区域）和一个提示，你可以修改图像的部分内容——例如，给我们的兔子加上一顶帽子。

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# 编辑也作为base64返回
edited_image = base64.b64decode(response.data[0].b64_json)
```

基础图像只包含兔子；最终图像中兔子戴上了帽子。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
